<a href="https://colab.research.google.com/github/averoo/projek-akhir/blob/main/demo%20program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import deque

class SistemManajemenKlinik:
    def __init__(self):
        # --- BAGIAN 3.1: MANAJEMEN PASIEN ---
        # STRUKTUR DATA: Dictionary (Hash Map)
        # JUSTIFIKASI: Memungkinkan pencarian dan pembaruan data pasien berdasarkan Nomor ID
        # dengan kompleksitas waktu O(1)[cite: 20, 22, 52].
        self.pasien_db = {}
        self.counter_id_pasien = 1

        # --- BAGIAN 3.2: MANAJEMEN ANTRIAN ---
        # STRUKTUR DATA: Deque (Double-Ended Queue) dari library bawaan Python
        # JUSTIFIKASI: Operasi penambahan di akhir (append) dan penghapusan di awal (popleft)
        # berjalan dalam waktu O(1), sangat efisien untuk antrian FIFO[cite: 27, 43, 52].
        self.antrian_prioritas = deque()
        self.antrian_reguler = deque()
        self.counter_prioritas = 1
        self.counter_reguler = 1

        # --- BAGIAN 3.3: REKAM MEDIS & RIWAYAT ---
        # STRUKTUR DATA: Dictionary berisikan List yang berfungsi sebagai STACK
        # JUSTIFIKASI: Operasi pembatalan tindakan terakhir (Undo) sangat tepat menggunakan
        # prinsip LIFO (Last In, First Out) dengan fungsi append() dan pop() bernilai O(1)[cite: 31, 52].
        self.riwayat_tindakan = {}

        # --- BAGIAN 3.4: LAPORAN & STATISTIK HARIAN ---
        self.total_pasien_dilayani = 0
        self.log_transaksi = [] # List biasa untuk mencatat riwayat transaksi kronologis [cite: 35]


    # =================================================================================
    # FITUR 3.1: MANAJEMEN PASIEN & OTOMATISASI ANTRIAN (3.2)
    # =================================================================================
    def daftarkan_pasien_baru(self, nama, usia, keluhan, is_darurat=False):
        """Mendaftarkan pasien baru ke database klinik dan memasukkannya ke antrian[cite: 19, 25]."""
        # Pembuatan Nomor ID Pasien Unik Otomatis
        id_pasien = f"ID-{self.counter_id_pasien:03d}"
        self.counter_id_pasien += 1

        # Simpan ke Database Pasien [cite: 19]
        self.pasien_db[id_pasien] = {
            "nama": nama,
            "usia": usia,
            "keluhan": keluhan,
            "is_darurat": is_darurat
        }
        print(f"\n✅ [BERHASIL] Pasien terdaftar dengan Nomor ID Pasien: {id_pasien}")

        # Ketentuan Antrian Prioritas: Pasien darurat ATAU lansia (usia > 60) [cite: 26]
        pasien_antrian = {"id_pasien": id_pasien, "nama": nama, "usia": usia}

        if is_darurat or len(nama) == 0 or usia > 60:
            pasien_antrian["nomor_antrian"] = f"P-{self.counter_prioritas:03d}"
            self.antrian_prioritas.append(pasien_antrian)
            self.counter_prioritas += 1
            print(f"👉 {nama} dimasukkan ke Antrian PRIORITAS ({pasien_antrian['nomor_antrian']}) [cite: 26]")
        else:
            # Antrian Reguler: Pasien umum yang mendaftar secara normal [cite: 26]
            pasien_antrian["nomor_antrian"] = f"R-{self.counter_reguler:03d}"
            self.antrian_reguler.append(pasien_antrian)
            self.counter_reguler += 1
            print(f"👉 {nama} dimasukkan ke Antrian REGULER ({pasien_antrian['nomor_antrian']}) [cite: 26]")

    def cari_data_pasien(self, id_pasien):
        """Mencari data pasien berdasarkan nomor ID dengan cepat O(1)."""
        if id_pasien in self.pasien_db:
            p = self.pasien_db[id_pasien]
            print(f"\n📄 --- DATA PASIEN ({id_pasien}) ---")
            print(f"Nama    : {p['nama']}")
            print(f"Usia    : {p['usia']} tahun")
            print(f"Keluhan : {p['keluhan']}")
            return p
        else:
            print("❌ Error: Nomor ID pasien tidak ditemukan di sistem! [cite: 38]")
            return None

    def tampilkan_seluruh_pasien(self):
        """Menampilkan daftar seluruh pasien yang pernah terdaftar[cite: 21]."""
        print("\n📋 --- DAFTAR SELURUH PASIEN TERDAFTAR ---")
        if not self.pasien_db:
            print("(Belum ada pasien yang terdaftar di database)")
            return
        for id_p, data in self.pasien_db.items():
            status = "Prioritas" if (data['is_darurat'] or data['usia'] > 60) else "Reguler"
            print(f"- ID: {id_p} | Nama: {data['nama']} | Usia: {data['usia']} thn | Status: {status}")

    def perbarui_data_pasien(self, id_pasien, nama_baru, usia_baru, keluhan_baru):
        """Memperbarui data pasien yang sudah ada berdasarkan ID[cite: 22]."""
        if id_pasien in self.pasien_db:
            self.pasien_db[id_pasien]["nama"] = nama_baru
            self.pasien_db[id_pasien]["usia"] = usia_baru
            self.pasien_db[id_pasien]["keluhan"] = keluhan_baru
            print(f"✅ Data pasien dengan ID {id_pasien} berhasil diperbarui! [cite: 22]")
        else:
            print("❌ Error: Gagal memperbarui, ID pasien tidak ditemukan! [cite: 38]")


    # =================================================================================
    # FITUR 3.2: MANAJEMEN ANTRIAN
    # =================================================================================
    def panggil_pasien_berikutnya(self):
        """Memanggil pasien berikutnya dengan mendahulukan Antrian Prioritas[cite: 27]."""
        print("\n📢 [PANGGILAN ANTRIAN]")

        # Mendahulukan Antrian Prioritas [cite: 27]ya5tida
        if self.antrian_prioritas:
            pasien = self.antrian_prioritas.popleft()
            p_data = self.pasien_db[pasien["id_pasien"]]
            keterangan = "Darurat" if p_data["is_darurat"] else f"Lansia ({pasien['usia']} tahun)"
            print(f"🔊 Nomor {pasien['nomor_antrian']} atas nama {pasien['nama']} ({pasien['id_pasien']})")
            print(f"   Kategori: PRIORITAS ({keterangan}) -> Silakan menuju Ruang Pemeriksaan Utama. [cite: 27]")
            return pasien["id_pasien"]

        # Jika prioritas kosong, panggil reguler [cite: 27]
        elif self.antrian_reguler:
            pasien = self.antrian_reguler.popleft()
            print(f"🔊 Nomor {pasien['nomor_antrian']} atas nama {pasien['nama']} ({pasien['id_pasien']})")
            print(f"   Kategori: REGULER -> Silakan menuju Ruang Pemeriksaan. [cite: 27]")
            return pasien["id_pasien"]

        else:
            print("📭 Semua antrian saat ini kosong. [cite: 27]")
            return None

    def tampilkan_kondisi_antrian(self):
        """Menampilkan kondisi antrian saat ini (jumlah dan urutan pasien)[cite: 27]."""
        print("\n============================ STATUS ANTRIAN SAAT INI ============================")

        # 1. Tampilkan Antrian Prioritas [cite: 27]
        print(f"🔴 ANTRIAN PRIORITAS (Total: {len(self.antrian_prioritas)} pasien)")
        if not self.antrian_prioritas:
            print("   (Kosong)")
        else:
            for indeks, pasien in enumerate(self.antrian_prioritas, 1):
                p_data = self.pasien_db[pasien["id_pasien"]]
                kategori = "Darurat" if p_data["is_darurat"] else "Lansia"
                print(f"   {indeks}. [{pasien['nomor_antrian']}] {pasien['nama']} ({pasien['id_pasien']}) - {kategori}")

        print("-" * 80)

        # 2. Tampilkan Antrian Reguler [cite: 27]
        print(f"🔵 ANTRIAN REGULER (Total: {len(self.antrian_reguler)} pasien)")
        if not self.antrian_reguler:
            print("   (Kosong)")
        else:
            for indeks, pasien in enumerate(self.antrian_reguler, 1):
                print(f"   {indeks}. [{pasien['nomor_antrian']}] {pasien['nama']} ({pasien['id_pasien']})")

        print("=================================================================================\n")


    # =================================================================================
    # FITUR 3.3: REKAM MEDIS & RIWAYAT (STACK IMPLEMENTATION)
    # =================================================================================
    def catat_tindakan_medis(self, id_pasien, diagnosa, resep, tindakan, biaya):
        """Mencatat tindakan medis pasien ke dalam Stack[cite: 29]."""
        if id_pasien not in self.pasien_db:
            print("❌ Error: Pasien tidak ditemukan di database! Tindakan medis dibatalkan. [cite: 38]")
            return

        if id_pasien not in self.riwayat_tindakan:
            self.riwayat_tindakan[id_pasien] = []

        # PUSH ke dalam Stack [cite: 29]
        catatan = {"diagnosa": diagnosa, "resep": resep, "tindakan": tindakan, "biaya": biaya}
        self.riwayat_tindakan[id_pasien].append(catatan)

        # Catat transaksi keuangan & statistik [cite: 33, 35]
        self.log_transaksi.append({"id_pasien": id_pasien, "nominal": biaya})
        self.total_pasien_dilayani += 1

        print(f"✅ Tindakan medis untuk {self.pasien_db[id_pasien]['nama']} berhasil disimpan! [cite: 29]")

    def tampilkan_riwayat_terakhir(self, id_pasien):
        """Menampilkan rekam medis paling terakhir (PEEK pada Stack)[cite: 30]."""
        if id_pasien not in self.riwayat_tindakan or not self.riwayat_tindakan[id_pasien]:
            print("⚠️ Belum ada riwayat tindakan medis untuk pasien ini. [cite: 30]")
            return

        # Mengakses elemen paling akhir tanpa menghapusnya (Peek)
        terakhir = self.riwayat_tindakan[id_pasien][-1]
        print(f"\n⚕️ --- RIWAYAT MEDIS TERAKHIR PASIEN ({id_pasien}) --- [cite: 30]")
        print(f"Diagnosa : {terakhir['diagnosa']}")
        print(f"Resep    : {terakhir['resep']}")
        print(f"Tindakan : {terakhir['tindakan']}")

    def batalkan_tindakan_terakhir(self, id_pasien):
        """Membatalkan (Undo) rekaman medis paling terakhir (POP dari Stack)."""
        if id_pasien not in self.riwayat_tindakan or not self.riwayat_tindakan[id_pasien]:
            print("❌ Error: Tidak ada tindakan yang bisa dibatalkan untuk pasien ini! [cite: 31, 38]")
            return

        # POP dari Stack untuk pembatalan LIFO
        dibatalkan = self.riwayat_tindakan[id_pasien].pop()

        # Sinkronisasi: hapus log pembayaran terakhir milik pasien tersebut dari laporan harian
        for i in reversed(range(len(self.log_transaksi))):
            if self.log_transaksi[i]["id_pasien"] == id_pasien:
                self.log_transaksi.pop(i)
                self.total_pasien_dilayani -= 1
                break

        print(f"✅ [UNDO BERHASIL] Tindakan terakhir ({dibatalkan['diagnosa']}) pasien {id_pasien} telah dihapus! ")


    # =================================================================================
    # FITUR 3.4: LAPORAN & STATISTIK HARIAN
    # =================================================================================
    def tampilkan_laporan_harian(self, kriteria_sort):
        """Menampilkan total pelayanan, daftar terurut, dan log transaksi[cite: 32, 33, 34, 35]."""
        print("\n" + "="*40)
        print(" 📊 LAPORAN & STATISTIK HARIAN KLINIK ")
        print("="*40)
        print(f"Total Pasien Selesai Dilayani Hari Ini: {self.total_pasien_dilayani} pasien [cite: 33]")

        # Mengurutkan daftar pasien berdasarkan kriteria [cite: 34]
        print(f"\n--- Daftar Pasien Terdaftar (Urut Berdasarkan {kriteria_sort}) ---")
        if not self.pasien_db:
            print("Belum ada pasien yang terdaftar.")
        else:
            if kriteria_sort == "Nama":
                # Sorting Timsort O(n log n) berdasarkan Alphabet Nama
                pasien_terurut = sorted(self.pasien_db.items(), key=lambda x: x[1]["nama"].lower())
            else:
                # Sorting Timsort O(n log n) berdasarkan ID Pasien
                pasien_terurut = sorted(self.pasien_db.items(), key=lambda x: x[0])

            for id_p, data in pasien_terurut:
                print(f"- ID: {id_p} | Nama: {data['nama']} | Usia: {data['usia']} thn")

        # Riwayat log transaksi pembayaran [cite: 35]
        print("\n--- Log Transaksi Keuangan Hari Ini ---")
        if not self.log_transaksi:
            print("Belum ada transaksi pembayaran masuk. [cite: 35]")
        else:
            total_pemasukan = 0
            for t in self.log_transaksi:
                print(f"- Pasien: {t['id_pasien']} | Nominal: Rp{t['nominal']:,}")
                total_pemasukan += t['nominal']
            print(f"💰 Total Pemasukan Hari Ini: Rp{total_pemasukan:,}")


# =================================================================================
# FITUR 3.5: ANTARMUKA TERMINAL INTERAKTIF (MAIN RUNNER)
# =================================================================================
if __name__ == "__main__":
    klinik = SistemManajemenKlinik()

    while True: # Loop berjalan terus menerus hingga memilih keluar [cite: 39]
        print("\n" + "="*45)
        print("   SISTEM INTEGRASI KLINIK SEHAT BERSAMA   ")
        print("="*45)
        print("[1] Daftarkan Pasien Baru (Otomatis Antrian) [cite: 81]")
        print("[2] Panggil Pasien Berikutnya [cite: 83]")
        print("[3] Cari Data Pasien Berdasarkan ID [cite: 85]")
        print("[4] Perbarui (Update) Data Pasien [cite: 22]")
        print("[5] Tampilkan Seluruh Pasien Terdaftar [cite: 21]")
        print("[6] Catat Tindakan Medis & Biaya [cite: 86]")
        print("[7] Lihat Riwayat Medis Terakhir Pasien [cite: 30]")
        print("[8] Batalkan Tindakan Terakhir Pasien (Undo) [cite: 87]")
        print("[9] Tampilkan Antrian Saat Ini [cite: 88]")
        print("[10] Laporan & Statistik Harian Klinik [cite: 89]")
        print("[0] Keluar Aplikasi [cite: 90]")
        print("="*45)

        pilihan = input("Pilih nomor menu (0-10): ") # Interaksi menggunakan angka [cite: 37]

        # Validasi Input Angka Menu Utama [cite: 38]
        if pilihan == '1':
            nama = input("Masukkan Nama Pasien: ")
            # Loop validasi input angka usia agar tidak crash [cite: 38]
            while True:
                try:
                    usia = int(input("Masukkan Usia Pasien: "))
                    break
                except ValueError:
                    print("❌ Error: Usia harus berupa angka bulat positif! Silakan coba lagi. [cite: 38]")
            keluhan = input("Keluhan Medis Pasien: ")
            is_darurat_in = input("Apakah pasien kondisi darurat? (ya/tidak): ").strip().lower()
            is_darurat = True if is_darurat_in == 'ya' else False

            klinik.daftarkan_pasien_baru(nama, usia, keluhan, is_darurat)

        elif pilihan == '2':
            klinik.panggil_pasien_berikutnya()

        elif pilihan == '3':
            id_cari = input("Masukkan ID Pasien yang dicari (contoh: ID-001): ").strip().upper()
            klinik.cari_data_pasien(id_cari)

        elif pilihan == '4':
            id_up = input("Masukkan ID Pasien yang ingin diperbarui: ").strip().upper()
            if id_up in klinik.pasien_db:
                nama_n = input("Masukkan Nama Baru: ")
                while True:
                    try:
                        usia_n = int(input("Masukkan Usia Baru: "))
                        break
                    except ValueError:
                        print("❌ Error: Usia harus berupa angka bulat! [cite: 38]")
                keluhan_n = input("Masukkan Keluhan Baru: ")
                klinik.perbarui_data_pasien(id_up, nama_n, usia_n, keluhan_n)
            else:
                print("❌ Error: ID Pasien tidak ditemukan! [cite: 38]")

        elif pilihan == '5':
            klinik.tampilkan_seluruh_pasien()

        elif pilihan == '6':
            id_medis = input("Masukkan ID Pasien yang diperiksa: ").strip().upper()
            if id_medis in klinik.pasien_db:
                diagnosa = input("Masukkan Diagnosa Medis: ")
                resep = input("Masukkan Resep Obat: ")
                tindakan = input("Masukkan Tindakan yang Diberikan: ")
                while True:
                    try:
                        biaya = int(input("Masukkan Total Nominal Biaya Pembayaran: Rp"))
                        break
                    except ValueError:
                        print("❌ Error: Nominal biaya harus berupa angka bulat! [cite: 38]")
                klinik.catat_tindakan_medis(id_medis, diagnosa, resep, tindakan, biaya)
            else:
                print("❌ Error: ID Pasien tidak ditemukan! [cite: 38]")

        elif pilihan == '7':
            id_view = input("Masukkan ID Pasien: ").strip().upper()
            klinik.tampilkan_riwayat_terakhir(id_view)

        elif pilihan == '8':
            id_undo = input("Masukkan ID Pasien yang ingin dibatalkan tindakan terakhirnya: ").strip().upper()
            klinik.batalkan_tindakan_terakhir(id_undo)

        elif pilihan == '9':
            klinik.tampilkan_kondisi_antrian()

        elif pilihan == '10':
            print("\nPilih Kriteria Pengurutan Daftar Pasien:")
            print("[A] Urutkan Alfabetis Nama")
            print("[B] Urutkan Berdasarkan ID Pasien")
            krt = input("Pilih kriteria (A/B): ").strip().upper()

            if krt == 'A':
                klinik.tampilkan_laporan_harian("Nama")
            elif krt == 'B':
                klinik.tampilkan_laporan_harian("ID Pasien")
            else:
                print("❌ Pilihan kriteria tidak valid. Menampilkan urutan default (ID). [cite: 38]")
                klinik.tampilkan_laporan_harian("ID Pasien")

        elif pilihan == '0':
            print("\nTerima kasih! Keluar dari Sistem Manajemen Klinik Sehat Bersama.")
            break # Hentikan program (loop break) [cite: 39]

        else:
            print("❌ Error: Pilihan menu tidak valid! Masukkan angka antara 0 - 10. [cite: 38]")


   SISTEM INTEGRASI KLINIK SEHAT BERSAMA   
[1] Daftarkan Pasien Baru (Otomatis Antrian) [cite: 81]
[2] Panggil Pasien Berikutnya [cite: 83]
[3] Cari Data Pasien Berdasarkan ID [cite: 85]
[4] Perbarui (Update) Data Pasien [cite: 22]
[5] Tampilkan Seluruh Pasien Terdaftar [cite: 21]
[6] Catat Tindakan Medis & Biaya [cite: 86]
[7] Lihat Riwayat Medis Terakhir Pasien [cite: 30]
[8] Batalkan Tindakan Terakhir Pasien (Undo) [cite: 87]
[9] Tampilkan Antrian Saat Ini [cite: 88]
[10] Laporan & Statistik Harian Klinik [cite: 89]
[0] Keluar Aplikasi [cite: 90]
Pilih nomor menu (0-10): 1
Masukkan Nama Pasien: Budi
Masukkan Usia Pasien: 50
Keluhan Medis Pasien: Sakit Hati
Apakah pasien kondisi darurat? (ya/tidak): ya

✅ [BERHASIL] Pasien terdaftar dengan Nomor ID Pasien: ID-001
👉 Budi dimasukkan ke Antrian PRIORITAS (P-001) [cite: 26]

   SISTEM INTEGRASI KLINIK SEHAT BERSAMA   
[1] Daftarkan Pasien Baru (Otomatis Antrian) [cite: 81]
[2] Panggil Pasien Berikutnya [cite: 83]
[3] Cari Data Pasien B